In [ ]:
import os
import yaml
import polars as pl
import pandas as pd
import numpy as np
from tqdm import tqdm
from plotnine import *

from scripts import get_correlations

In [ ]:
config_path = "/home/dnanexus/ukbgym/config_wgs.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)


all_annotation_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        all_annotation_list.extend(category)

all_annotation_list = list(set(all_annotation_list))
len(all_annotation_list)

In [ ]:
rare_variant_annotations_dict = config.get('rare_variant_annotations')
# Drop the 'misc' key if present
rare_variant_annotations_dict = {k: v for k, v in rare_variant_annotations_dict.items() if k != 'misc'}
rare_variant_annotations_dict

## Merge pos-neg split annotations with other annotations

In [ ]:
config_path = "/home/dnanexus/ukbgym/config_wgs_cadd.yaml"

with open(config_path) as f:
    config = yaml.safe_load(f)


pos_neg_annos_list = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
if rare_variant_annotations_dict:
    for category in rare_variant_annotations_dict.values():
        pos_neg_annos_list.extend(category)

pos_neg_annos_list = list(set(pos_neg_annos_list))
len(pos_neg_annos_list)

In [ ]:
del_annos = list(set([an[:-4] for an in pos_neg_annos_list]))
len(del_annos)

In [ ]:
burdens_path = "/home/dnanexus/data_dir/burdens"

all_annos_files = os.listdir(burdens_path + '/55_small_genes_onlySNP_cadd_annotations')

for gene_file in tqdm(all_annos_files):
    print(f"Processing {gene_file}...")

    bdf = pl.read_parquet(
        burdens_path + f'/55_small_genes_onlySNP_cadd_annotations/{gene_file}'
    ).filter(~pl.col('annotation').is_in(del_annos))

    pndf = pl.read_parquet(
        burdens_path + f'/55_small_genes_onlySNP_cadd_annotations_posnegsplit/{gene_file}'
    )

    pl.concat([bdf, pndf]).lazy().sink_parquet(
        burdens_path + f'/55_small_genes_onlySNP_cadd_annotations_posnegsplit_merged/{gene_file}'
    )


## Debug compute correlations

In [ ]:
assocs = pl.read_parquet('/home/dnanexus/data_dir/small_gene_assocs.pq').with_columns(
    (pl.col('phenotype') + '_prs_corrected').alias('phenotype')
)
assocs

In [ ]:
pheno_dir = '/home/dnanexus/data_dir/phenotypes_corr'
pheno_df = pl.concat([pl.read_parquet(f'{pheno_dir}/{f}') for f in os.listdir(pheno_dir) if f.endswith('.parquet')], how='align').unpivot(index=['sample'], variable_name='phenotype', value_name='value').rename({'sample': 'sample_id'})
pheno_df

In [ ]:
burdens_dir = "/home/dnanexus/data_dir/burdens/55_small_genes_onlySNP_cadd_annotations_posnegsplit_merged"
gene_file = 'ENSG00000130164.parquet'

bdf = pl.read_parquet(burdens_dir + f'/{gene_file}')
bdf

In [ ]:
adf = assocs.filter(pl.col('gene_id') == gene_file.split('.')[0])
adf

In [ ]:
subset_annos = ['CADD_PHRED', 'loftee_hc', 'am_pathogenicity', 'pangolin_score', 'gpn_score_neg', 'CADD_SIFTval', 'AbSplice2_max']

cdf = bdf.filter(pl.col('annotation').is_in(subset_annos)).join(pheno_df.filter(pl.col('phenotype').is_in(adf['phenotype'])), on='sample_id')
cdf

In [ ]:
def compute_correlations_lazy(df_lazy: pl.LazyFrame) -> pl.LazyFrame:
    """
    Computes both Pearson and Spearman correlations:
    - Between ('sum', 'value'), ('max', 'value'), ('top2', 'value')
    - Grouped by ('annotation', 'phenotype')
    """
    # Filter out NaN & null
    df_clean = df_lazy.filter(
        pl.all_horizontal(
            pl.col(["sum", "max", "top2", "value"]).is_not_nan() & pl.col(["sum", "max", "top2", "value"]).is_not_null()
        )
    )

    # Add ranks per group for Spearman
    df_ranks = df_clean.with_columns([
        pl.col("sum").rank().over(["annotation", "phenotype"]).alias("rank_sum"),
        pl.col("max").rank().over(["annotation", "phenotype"]).alias("rank_max"),
        pl.col("top2").rank().over(["annotation", "phenotype"]).alias("rank_top2"),
        pl.col("value").rank().over(["annotation", "phenotype"]).alias("rank_value"),
    ])

    # Group by + compute both sets of correlations
    correlations = df_ranks.group_by(["annotation", "phenotype"]).agg([
        # Pearson
        pl.corr("sum", "value").alias("sum_pearson"),
        pl.corr("max", "value").alias("max_pearson"),
        pl.corr("top2", "value").alias("top2_pearson"),
        # Spearman
        pl.corr("rank_sum", "rank_value").alias("sum_spearman"),
        pl.corr("rank_max", "rank_value").alias("max_spearman"),
        pl.corr("rank_top2", "rank_value").alias("top2_spearman"),
    ])

    return correlations

In [ ]:
rank_corr_df = compute_correlations_lazy(cdf.lazy()).collect()
rank_corr_df

In [ ]:
rank_corr_df.filter(pl.col('annotation') == 'am_pathogenicity')

In [ ]:
annotation_category_map = {}
if rare_variant_annotations_dict:
    for category, annotations in rare_variant_annotations_dict.items():
        for ann in annotations:
            annotation_category_map[ann] = category

# Add category
corr_df = rank_corr_df.with_columns(
    pl.col("annotation").replace(annotation_category_map).alias("category")
)

# Fill NaNs
corr_cols = [col for col in corr_df.columns if "pearson" in col or "spearman" in col]
corr_df = corr_df.with_columns(
    [pl.col(col).fill_null(0).alias(col) for col in corr_cols]
)

# Melt
corr_long = corr_df.unpivot(
    index=["annotation", "phenotype", "category"],
    variable_name="correlation_type",
    value_name="correlation"
).with_columns([
    pl.col("correlation_type").str.extract(r"(pearson|spearman)").alias("method"),
    pl.col("correlation_type").str.extract(r"(sum|max|top2)").alias("aggregation"),
    pl.col("correlation").abs().alias("abs_correlation")
])
corr_long

In [ ]:
aggregation = "sum"
method = "spearman"

agg_df = (
    corr_long
    .filter(pl.col("aggregation") == aggregation)
    .filter(pl.col("method") == method)
    .group_by("annotation")
    .agg(pl.median("abs_correlation").alias("median_abs_correlation"))
    .sort("median_abs_correlation", descending=True)
)

ordered_annotations = agg_df['annotation'].to_list()

corr_long_pd = corr_long.to_pandas()
corr_long_pd['annotation'] = pd.Categorical(
    corr_long_pd['annotation'],
    categories=ordered_annotations,
    ordered=True
)

(
    ggplot(
        corr_long_pd.query(f"aggregation == '{aggregation}' & method == '{method}'"),
        aes(x='annotation', y='abs_correlation', fill='category')
    )
    + geom_boxplot(alpha=0.75)
    + theme_bw()
    + scale_y_sqrt()
    + ylab('|rank correlation|')
    + theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(12, 8),
    )
)

## Compute correlations

In [ ]:
def compute_correlations_lazy(df_lazy: pl.LazyFrame) -> pl.LazyFrame:
    """
    Computes both Pearson and Spearman correlations:
    - Between ('sum', 'value'), ('max', 'value'), ('top2', 'value')
    - Grouped by ('annotation', 'phenotype')
    """
    # Filter out NaN & null
    df_clean = df_lazy.filter(
        pl.all_horizontal(
            pl.col(["sum", "max", "top2", "value"]).is_not_nan() & pl.col(["sum", "max", "top2", "value"]).is_not_null()
        )
    )

    # Add ranks per group for Spearman
    df_ranks = df_clean.with_columns([
        pl.col("sum").rank().over(["annotation", "phenotype"]).alias("rank_sum"),
        pl.col("max").rank().over(["annotation", "phenotype"]).alias("rank_max"),
        pl.col("top2").rank().over(["annotation", "phenotype"]).alias("rank_top2"),
        pl.col("value").rank().over(["annotation", "phenotype"]).alias("rank_value"),
    ])

    # Group by + compute both sets of correlations
    correlations = df_ranks.group_by(["annotation", "phenotype"]).agg([
        # Pearson
        pl.corr("sum", "value").alias("sum_pearson"),
        pl.corr("max", "value").alias("max_pearson"),
        pl.corr("top2", "value").alias("top2_pearson"),
        # Spearman
        pl.corr("rank_sum", "rank_value").alias("sum_spearman"),
        pl.corr("rank_max", "rank_value").alias("max_spearman"),
        pl.corr("rank_top2", "rank_value").alias("top2_spearman"),
    ])

    return correlations

In [ ]:
data_dir = '/home/dnanexus/data_dir'
pheno_dir = f'{data_dir}/phenotypes_corr'
burdens_dir = f"{data_dir}/burdens/55_small_genes_onlySNP_cadd_annotations_posnegsplit_merged"

# Subset to a smaller set of annotations
subset_annos = []
rare_variant_annotations_dict = config.get('rare_variant_annotations')
rare_variant_annotations_dict = {k: v for k, v in rare_variant_annotations_dict.items() if k != 'misc'}
for category in rare_variant_annotations_dict.values():
    subset_annos.extend(category)
subset_annos = list(set(subset_annos))

assocs = pl.read_parquet(f'{data_dir}/small_gene_assocs.pq').with_columns(
    (pl.col('phenotype') + '_prs_corrected').alias('phenotype')
)

# Read once, up front
pheno_df = (
    pl.concat(
        [pl.read_parquet(f'{pheno_dir}/{f}')
         for f in os.listdir(pheno_dir) if f.endswith('.parquet')],
        how='align'
    )
    .unpivot(index=['sample'],
             variable_name='phenotype',
             value_name='value')
    .rename({'sample': 'sample_id'})
)

corr_df_list = []
# In loop: filter using an eager list, then switch to lazy for join
for gene_file in tqdm(os.listdir(burdens_dir), desc="Correlations for genes"):
    if not gene_file.endswith('.parquet'):
        continue

    gene_id = gene_file.split('.')[0]
    adf = assocs.filter(pl.col('gene_id') == gene_id)

    phenos_needed = adf['phenotype'].unique().to_list()

    pheno_filtered = (
        pheno_df
        .filter(pl.col('phenotype').is_in(phenos_needed))
        .lazy()
    )

    bdf = pl.scan_parquet(f'{burdens_dir}/{gene_file}').filter(pl.col('annotation').is_in(subset_annos))

    cdf = bdf.join(pheno_filtered, on='sample_id', how='inner')

    corr_df_list.append(
        compute_correlations_lazy(cdf).with_columns(
            pl.lit(gene_id).alias('gene_id')
        ).collect()
    )

corr_df = pl.concat(corr_df_list)

In [ ]:
corr_df

In [ ]:
annotation_category_map = {}
if rare_variant_annotations_dict:
    for category, annotations in rare_variant_annotations_dict.items():
        for ann in annotations:
            annotation_category_map[ann] = category

# Add category
corr_df = corr_df.with_columns(
    pl.col("annotation").replace(annotation_category_map).alias("category")
)

# Fill NaNs
corr_cols = [col for col in corr_df.columns if "pearson" in col or "spearman" in col]
corr_df = corr_df.with_columns(
    [pl.col(col).fill_null(0).alias(col) for col in corr_cols]
)

# Melt
corr_long = corr_df.unpivot(
    index=["annotation", "phenotype", "gene_id", "category"],
    variable_name="correlation_type",
    value_name="correlation"
).with_columns([
    pl.col("correlation_type").str.extract(r"(pearson|spearman)").alias("method"),
    pl.col("correlation_type").str.extract(r"(sum|max|top2)").alias("aggregation"),
    pl.col("correlation").abs().alias("abs_correlation")
])
corr_long

In [ ]:
ordered_annotations = agg_df['annotation'].to_list()
ordered_annotations

In [ ]:
aggregation = "top2"
method = "spearman"

agg_df = (
    corr_long
    .filter(pl.col("aggregation") == aggregation)
    .filter(pl.col("method") == method)
    .group_by("annotation")
    .agg(pl.median("abs_correlation").alias("median_abs_correlation"))
    .sort("median_abs_correlation", descending=True)
)

ordered_annotations = agg_df['annotation'].to_list()

corr_long_pd = corr_long.to_pandas()
corr_long_pd['annotation'] = pd.Categorical(
    corr_long_pd['annotation'],
    categories=ordered_annotations,
    ordered=True
)

(
    ggplot(
        corr_long_pd.query(f"aggregation == '{aggregation}' & method == '{method}'"),
        aes(x='annotation', y='abs_correlation', fill='category')
    )
    + geom_boxplot(alpha=0.75)
    + theme_bw()
    + scale_y_sqrt()
    + ylab('|rank correlation|')
    + theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(15, 8),
    )
)